In [2]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "code"))

from mdp.task import Task
from mdp.catalogue import Catalogue
from mdp.state import Specification, Term
from mdp.action import *
from env.apollo.generator import ApolloGenerator

from env.result_cache import ResultCache

## Test rewards

1. Create a temporary rewards database

In [3]:
cache_path = Path("./test_rewards.sqlite")

if cache_path.exists():
    cache_path.unlink()

cache = ResultCache(cache_path)

cache

ResultCache(db_path=PosixPath('test_rewards.sqlite'))

2. Create a failed outcome

In [7]:
failed = cache.failed(task_name="apollo_mode_choice", specification="spec_1")
failed

,task_name,specification,numParams,numResids,maximum,vcHessianConditionNumber,successfulEstimation,LL0,LLC,LLout,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,apollo_mode_choice,spec_1,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0


3. Create a skipped outcome

In [9]:
skipped = cache.skipped(task_name="apollo_mode_choice", specification="spec_2", n_free_parameters=60)
skipped

,task_name,specification,numParams,numResids,maximum,vcHessianConditionNumber,successfulEstimation,LL0,LLC,LLout,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,apollo_mode_choice,spec_2,60,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,60,1


4. Add failed and skipped outcomes

In [10]:
cache.upsert(failed)
cache.upsert(skipped)

5. Check if a specification has already estimated 

In [13]:
cache.exists(task_name="apollo_mode_choice", specification="spec_1")

True

6. Retrieve modelling outcomes of a estimated model

In [12]:
cache.lookup(task_name="apollo_mode_choice", specification="spec_1")

,task_name,specification,numParams,numResids,maximum,vcHessianConditionNumber,successfulEstimation,LL0,LLC,LLout,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,apollo_mode_choice,spec_1,None,None,None,None,0,None,None,None,None,None,None,None,None,None,None,None,None,0


7. Load all the model outcomes

In [15]:
cache.load()

,task_name,specification,numParams,numResids,maximum,vcHessianConditionNumber,successfulEstimation,LL0,LLC,LLout,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,apollo_mode_choice,spec_1,NaN,None,None,None,0,None,None,None,None,None,None,None,None,None,None,None,NaN,0
1,apollo_mode_choice,spec_2,60.0,None,None,None,0,None,None,None,None,None,None,None,None,None,None,None,60.0,1


8. update a model estimation outcome

In [16]:
updated = failed.copy()
updated["LLout"] = -1234
cache.upsert(updated)

cache.lookup(task_name="apollo_mode_choice", specification="spec_1")

,task_name,specification,numParams,numResids,maximum,vcHessianConditionNumber,successfulEstimation,LL0,LLC,LLout,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,apollo_mode_choice,spec_1,None,None,None,None,0,None,None,-1234.0,None,None,None,None,None,None,None,None,None,0


## Emprical rewards

In [25]:
cache_path = Path("../dataset/dataset_1/rewards.sqlite")
cache = ResultCache(cache_path)

cache.load()

,task_name,specification,numParams,numResids,maximum,vcHessianConditionNumber,successfulEstimation,LL0,LLC,LLout,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,ApolloModeChoice,1110_2110_3110_4110_5000_6110_7000_8000,7.0,5600.0,-4670.988194,5.802145e-07,1,-6607.093673,-5430.886635,-4670.988194,0.293034,0.291975,0.139922,0.139185,9355.976387,9402.390040,-25.901553,0.592869,7.0,0
1,ApolloModeChoice,1110_2110_3110_4110_5000_6120_7000_8000,8.0,5600.0,-4670.221880,6.592906e-07,1,-6607.093673,-5430.886635,-4670.221880,0.293150,0.291940,0.140063,0.139142,9356.443760,9409.487935,-20.973439,0.538391,8.0,0
2,ApolloModeChoice,1110_2110_3110_4110_5000_6210_7000_8000,7.0,5600.0,-4660.531979,4.405313e-07,1,-6607.093673,-5430.886635,-4660.531979,0.294617,0.293557,0.141847,0.141110,9335.063957,9381.477610,-20.114351,0.502049,7.0,0
3,ApolloModeChoice,1110_2110_3110_4110_5000_6220_7000_8000,8.0,5600.0,-4659.639354,2.441837e-07,1,-6607.093673,-5430.886635,-4659.639354,0.294752,0.293541,0.142011,0.141091,9335.278709,9388.322884,-10.116044,0.483173,8.0,0
4,ApolloModeChoice,1110_2110_3110_4110_5000_6310_7000_8000,8.0,5600.0,-4603.567575,1.183897e-36,1,-6607.093673,-5430.886635,-4603.567575,0.303239,0.302028,0.152336,0.151415,9223.135150,9276.179325,NaN,0.367627,8.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26385,ApolloModeChoice,1110_2310_3112_4000_5120_6000_7000_8000,9.0,5600.0,-4563.285215,2.534571e-06,1,-6607.093673,-5430.886635,-4563.285215,0.309335,0.307973,0.159753,0.158648,9144.570430,9204.245127,-24.059556,0.715114,9.0,0
26386,ApolloModeChoice,1116_2110_3216_4000_5120_6000_7000_8000,19.0,5600.0,-4698.099622,6.077046e-06,1,-6607.093673,-5430.886635,-4698.099622,0.288931,0.286055,0.134930,0.131983,9434.199244,9560.179160,-7.391760,1.554350,19.0,0
26387,ApolloModeChoice,1112_2110_3216_4000_5216_6000_7000_8000,15.0,5600.0,-4346.970751,6.479791e-06,1,-6607.093673,-5430.886635,-4346.970751,0.342075,0.339805,0.199584,0.197374,8723.941502,8823.399330,-6.136886,1.213176,15.0,0
26388,ApolloModeChoice,1110_2316_3216_4000_5210_6000_7000_8000,13.0,5600.0,-4667.061211,1.074200e-05,1,-6607.093673,-5430.886635,-4667.061211,0.293629,0.291661,0.140645,0.138803,9360.122422,9446.319206,-25.088253,1.692919,13.0,0
